<a href="https://colab.research.google.com/github/rex9703/Minecraft-Mod-/blob/main/%E3%80%90Demo06b%E3%80%91RAG02_%E6%89%93%E9%80%A0_RAG_%E7%B3%BB%E7%B5%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 0. 讀入你打造好的 vector dataset

In [7]:
!pip -q install gdown

In [8]:
GDRIVE_PUBLIC_URL = "https://drive.google.com/file/d/1qpgWeuxJPSYfZDWFgi5MvI6C2EqFmNmb/view?usp=sharing"

In [9]:
!gdown --fuzzy -O faiss_db.zip "{GDRIVE_PUBLIC_URL}"

Downloading...
From: https://drive.google.com/uc?id=1qpgWeuxJPSYfZDWFgi5MvI6C2EqFmNmb
To: /content/faiss_db.zip
100% 17.4k/17.4k [00:00<00:00, 4.60MB/s]


In [10]:
!unzip faiss_db.zip

Archive:  faiss_db.zip
replace faiss_db/index.faiss? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

### 1. 安裝並引入必要套件

In [11]:
!pip install -U langchain langchain-community faiss-cpu transformers sentence-transformers huggingface_hub
!pip -q install "aisuite[all]"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... canceled
ERROR: Operation cancelled by user


In [12]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

/tmp/ipykernel_1735/2876051865.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [14]:
from openai import OpenAI
import gradio as gr

### 2. 自訂 E5 embedding 類別

In [15]:
import os
from google.colab import userdata

In [16]:
hf_token = userdata.get('HuggingFace')

In [17]:
from huggingface_hub import login
login(token=hf_token)

In [18]:
class EmbeddingGemmaEmbeddings(HuggingFaceEmbeddings):
    def __init__(self, **kwargs):
        super().__init__(
            model_name="google/embeddinggemma-300m",
            encode_kwargs={"normalize_embeddings": True},
            **kwargs
        )

    def embed_documents(self, texts):
        # 你也可以把 "none" 改成真實標題（檔名/章節名），效果會更穩
        texts = [f"title: none | text: {t}" for t in texts]
        return super().embed_documents(texts)

    def embed_query(self, text):
        # 官方檢索建議前綴
        return super().embed_query(f"task: search result | query: {text}")

### 3. 載入 `faiss_db`

In [20]:
embedding_model = EmbeddingGemmaEmbeddings()
vectorstore = FAISS.load_local(
    "faiss_db",
    embeddings=embedding_model,
    allow_dangerous_deserialization=True
)

modules.json:   0%|          | 0.00/573 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.21GB            

... (10 outputs left out)

2_Dense/model.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

3_Dense/model.safetensors: reconstructing file:   0%|          |  0.00B / 9.44MB            

3_Dense/model.safetensors: downloading bytes:           |  0.00B            

In [21]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

### 4. 設定好我們要的 LLM

如之前, 我們會用 OpenAI API。這裡使用 Groq 服務, 可改成你要的服務。

In [23]:
!pip install aisuite groq

  Using cached aisuite-0.2.0-py3-none-any.whl.metadata (12 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [aisuite]


In [24]:
import aisuite as ai

In [26]:
api_key = userdata.get('Groq_API_Key')

In [27]:
os.environ['GROQ_API_KEY']=api_key

這裡的模型和 `base_url` 是用 Groq, 如果用其他服務請自行修改。

In [28]:
model = "groq:openai/gpt-oss-120b"
#base_url="https://api.groq.com/openai/v1"

In [29]:
client = ai.Client()

### 5. `prompt` 設計

In [30]:
system_prompt = "你是一個名為「子浩的 NeoForge 伺服器精靈」的 AI 助理。你的職責是根據提供的伺服器規則與模組說明書，回答玩家關於伺服器設定、生存規範與模組用法的問題。請用幽默、友善、略帶遊戲魂的台灣繁體中文回答。"

prompt_template = """
根據下列檢索到的伺服器與模組文件：
{retrieved_chunks}

回答玩家的問題：{question}

注意：
1. 請務必只依據上述檢索到的文件回答。
2. 若資料不足以回答，請直接回覆：「這題超綱了！請直接在群組 tag 伺服主子浩處理啦！」
"""

### 6. 使用 RAG 來回應

搜尋與使用者問題相關的資訊，根據我們的 prompt 樣版去讓 LLM 回應。

In [31]:
chat_history = []

def chat_with_rag(user_input):
    global chat_history
    # 取回相關資料
    docs = retriever.get_relevant_documents(user_input)
    retrieved_chunks = "\n\n".join([doc.page_content for doc in docs])

    # 將自定 prompt 套入格式
    final_prompt = prompt_template.format(retrieved_chunks=retrieved_chunks, question=user_input)

    # 用 AI Suite 呼叫語言模型
    response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": final_prompt},
    ]
    )
    answer = response.choices[0].message.content

    chat_history.append((user_input, answer))
    return answer

### 7. 用 Gradio 打造 Web App

In [33]:
import gradio as gr

with gr.Blocks() as demo:
    gr.Markdown("# ⛏️ 子浩的 NeoForge 伺服器精靈")
    gr.ChatInterface(fn=chat_with_rag)

demo.launch(share=True)

/usr/local/lib/python3.13/dist-packages/gradio/utils.py:1379: UserWarning: Expected 1 arguments for function <function chat_with_rag at 0x79419fa67240>, received 2.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/gradio/utils.py:1387: UserWarning: Expected maximum 1 arguments for function <function chat_with_rag at 0x79419fa67240>, received 2.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/gradio/utils.py:1379: UserWarning: Expected 1 arguments for function <function chat_with_rag at 0x7941abfa4220>, received 2.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/gradio/utils.py:1387: UserWarning: Expected maximum 1 arguments for function <function chat_with_rag at 0x7941abfa4220>, received 2.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8c7595d554a4e440b3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
